# 08 — Online Model Inference Service (SPCS)

This notebook deploys the **tuned XGBoost demand forecast model** as a **real-time REST inference service** on Snowpark Container Services (SPCS). This enables:

- **Sub-second predictions** via a live HTTP endpoint
- **Auto-scaling** based on request volume
- **SQL integration** — call the service directly from SQL queries
- **Python integration** — call via `mv.run()` routed through the service

```
           ┌─────────────┐     ┌──────────────────┐     ┌──────────────┐
Request →  │ SPCS Gateway │ ──→ │ Model Container  │ ──→ │  Prediction  │
           │ (ingress)    │     │ (XGBoost TUNED)  │     │  Response    │
           └─────────────┘     └──────────────────┘     └──────────────┘
```

**Prerequisites:** Model `SNOWHEALTH_DEMAND_FORECAST` registered in `SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS` (from notebook 05).

In [ ]:
# Session setup
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
import pandas as pd
import numpy as np
import time

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}")

In [ ]:
# Verify the registered model and its predict function
from snowflake.ml.registry import Registry

VERSION_NAME = "TUNED_V1"

reg = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
model = reg.get_model(cfg.SNOWHEALTH_MODEL_NAME)
mv = model.version(VERSION_NAME)

functions = mv.show_functions()
print(f"Model: {cfg.SNOWHEALTH_MODEL_NAME} v{VERSION_NAME}")
print(f"Functions: {[f['name'] for f in functions]}")
print(f"\nPREDICT signature:")
for f in functions:
    if f['name'] == 'PREDICT':
        sig = f['signature']
        print(f"  Inputs ({len(sig.inputs)}): {[inp.name for inp in sig.inputs]}")
        print(f"  Outputs: {[o.name for o in sig.outputs]}")

In [ ]:
# Check available compute pools
pools_df = session.sql("SHOW COMPUTE POOLS").collect()

print("Available Compute Pools:")
for row in pools_df:
    row_dict = row.as_dict()
    cols = list(row_dict.keys())
    name = row_dict.get('name', row_dict.get(cols[0], '?'))
    family = row_dict.get('instance_family', row_dict.get(cols[4] if len(cols) > 4 else cols[0], '?'))
    state = row_dict.get('state', row_dict.get(cols[1] if len(cols) > 1 else cols[0], '?'))
    print(f"  {name:<30} {family:<16} {state}")

COMPUTE_POOL = "ML_DEMO_COMPUTE_POOL"
SERVICE_NAME = "SNOWHEALTH_DEMAND_INFERENCE_SERVICE"
print(f"\nSelected pool: {COMPUTE_POOL}")
print(f"Service name:  {SERVICE_NAME}")

In [ ]:
# Create the online inference service
print(f"Creating inference service: {SERVICE_NAME}")
print(f"  Model:    {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{cfg.SNOWHEALTH_MODEL_NAME} v{VERSION_NAME}")
print(f"  Pool:     {COMPUTE_POOL}")
print(f"  Ingress:  Enabled (HTTP endpoint)")
print(f"\nThis takes 5-10 minutes...\n")

# Drop existing service if present (safe for re-runs)
try:
    session.sql(f"DROP SERVICE IF EXISTS {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{SERVICE_NAME}").collect()
except:
    pass

mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL,
    ingress_enabled=True,
    max_instances=1,
)
print(f"Service created: {SERVICE_NAME}")

In [ ]:
# Monitor service startup until RUNNING
print("Monitoring service status...")
for attempt in range(30):
    status_json = session.sql(
        f"SELECT SYSTEM$GET_SERVICE_STATUS('{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{SERVICE_NAME}')"
    ).collect()[0][0]
    
    import json
    statuses = json.loads(status_json)
    container_status = statuses[0].get('status', 'UNKNOWN') if statuses else 'UNKNOWN'
    message = statuses[0].get('message', '') if statuses else ''
    
    print(f"  [{attempt+1}/30] Status: {container_status} - {message}")
    
    if container_status == 'READY':
        print(f"\nService is READY and accepting requests!")
        break
    elif container_status == 'FAILED':
        print(f"\nService FAILED. Fetching logs...")
        logs = session.sql(
            f"CALL SYSTEM$GET_SERVICE_LOGS('{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{SERVICE_NAME}', 0, 'model-inference')"
        ).collect()
        print(logs[0][0][-2000:] if logs else "No logs available")
        break
    else:
        time.sleep(20)
else:
    print("Service still starting — check manually with:")
    print(f"  SELECT SYSTEM$GET_SERVICE_STATUS('{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{SERVICE_NAME}')")

In [ ]:
# Get the service endpoint URL
endpoints = session.sql(
    f"SHOW ENDPOINTS IN SERVICE {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{SERVICE_NAME}"
).collect()

print("Service Endpoints:")
for row in endpoints:
    rd = row.as_dict()
    for k, v in rd.items():
        print(f"  {k}: {v}")
    print()

# Extract ingress URL
if endpoints:
    rd = endpoints[0].as_dict()
    ingress_key = [k for k in rd if 'ingress' in k.lower()]
    if ingress_key:
        print(f"Ingress URL: {rd[ingress_key[0]]}")
        print("This URL can be called from external applications with PAT authentication.")

In [ ]:
# Test 1: SQL-based inference through the service
# This is the simplest way to call the deployed model
print("Test 1: SQL Inference via Service")
print("=" * 60)

sql_result = session.sql(f"""
    SELECT {SERVICE_NAME}!PREDICT(
        UNIT_PRICE, PROMOTION_DISCOUNT_PCT, TEMPERATURE_F, FLU_INDEX,
        COMPETITOR_PROXIMITY_MILES, DAYS_SUPPLY_ON_HAND,
        MONTH_SIN, MONTH_COS, DOW_SIN, DOW_COS,
        ROLLING_7D_AVG_UNITS, ROLLING_30D_AVG_UNITS,
        LAG_7D_UNITS, LAG_14D_UNITS, LAG_30D_UNITS,
        PRICE_TIMES_PROMO, INVENTORY_TURNOVER_RATIO, SEASONAL_DEMAND_INDEX
    ) AS PREDICTION,
    SKU_ID, STORE_ID, UNITS_SOLD
    FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES
    LIMIT 10
""").to_pandas()

print(sql_result.to_string(index=False))
print(f"\nReturned {len(sql_result)} predictions via SQL")

In [ ]:
# Test 2: Python inference via mv.run() routed through the service
# This uses the same service endpoint but through the Python API
print("Test 2: Python Inference via mv.run() + Service")
print("=" * 60)

# Prepare test data (10 rows from feature table)
feature_input_cols = [
    'UNIT_PRICE', 'PROMOTION_DISCOUNT_PCT', 'TEMPERATURE_F', 'FLU_INDEX',
    'COMPETITOR_PROXIMITY_MILES', 'DAYS_SUPPLY_ON_HAND',
    'MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS',
    'ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS',
    'LAG_7D_UNITS', 'LAG_14D_UNITS', 'LAG_30D_UNITS',
    'PRICE_TIMES_PROMO', 'INVENTORY_TURNOVER_RATIO', 'SEASONAL_DEMAND_INDEX',
]

test_df = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES") \
    .select(feature_input_cols + ['SKU_ID', 'STORE_ID', 'UNITS_SOLD']).limit(10)

start = time.time()
result = mv.run(
    test_df.select(feature_input_cols),
    function_name="predict",
    service_name=SERVICE_NAME,
)
elapsed_ms = (time.time() - start) * 1000

result_pdf = result.to_pandas()
actuals = test_df.to_pandas()

print(f"Inference via service: {elapsed_ms:.0f} ms for {len(result_pdf)} rows\n")
comparison = pd.DataFrame({
    'SKU': actuals['SKU_ID'].values,
    'Store': actuals['STORE_ID'].values,
    'Actual': actuals['UNITS_SOLD'].values,
    'Predicted': result_pdf.iloc[:, -1].values.round(1),
})
comparison['Error'] = abs(comparison['Actual'] - comparison['Predicted']).round(1)
print(comparison.to_string(index=False))
print(f"\nMean absolute error: {comparison['Error'].mean():.1f} units")

In [ ]:
# Test 3: Latency benchmark — single-row predictions through the service
print("Test 3: Service Latency Benchmark")
print("=" * 60)

single_row = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES") \
    .select(feature_input_cols).limit(1)

latencies = []
for i in range(10):
    start = time.time()
    _ = mv.run(single_row, function_name="predict", service_name=SERVICE_NAME).collect()
    latencies.append((time.time() - start) * 1000)

latencies_arr = np.array(latencies)
print(f"Single-row inference via SPCS service ({len(latencies)} calls):")
print(f"  p50:  {np.percentile(latencies_arr, 50):.0f} ms")
print(f"  p95:  {np.percentile(latencies_arr, 95):.0f} ms")
print(f"  min:  {latencies_arr.min():.0f} ms")
print(f"  max:  {latencies_arr.max():.0f} ms")
print(f"  mean: {latencies_arr.mean():.0f} ms")

## Simulated Traffic Load Test
Sends repeated inference requests to the service endpoint to simulate real-world traffic patterns. Varies batch sizes and tracks throughput, latency percentiles, and error rates across the run.

In [ ]:
# Traffic simulation: send inference requests to the endpoint
# Simulates ~5 minutes of realistic API traffic with varying batch sizes
import time
import numpy as np
import pandas as pd

print("=" * 70)
print("TRAFFIC SIMULATION - SNOWHEALTH DEMAND INFERENCE SERVICE")
print("=" * 70)

# Config
TOTAL_DURATION_SEC = 300  # 5 minutes
BATCH_SIZES = [1, 5, 10, 25, 50]  # varying request sizes
PAUSE_BETWEEN_SEC = 2  # pause between requests to simulate real traffic

# Load feature data for inference
feature_input_cols = [
    'UNIT_PRICE', 'PROMOTION_DISCOUNT_PCT', 'TEMPERATURE_F', 'FLU_INDEX',
    'COMPETITOR_PROXIMITY_MILES', 'DAYS_SUPPLY_ON_HAND',
    'MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS',
    'ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS',
    'LAG_7D_UNITS', 'LAG_14D_UNITS', 'LAG_30D_UNITS',
    'PRICE_TIMES_PROMO', 'INVENTORY_TURNOVER_RATIO', 'SEASONAL_DEMAND_INDEX',
]

full_df = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES") \
    .select(feature_input_cols).to_pandas().astype(float)

print(f"Loaded {len(full_df)} rows for inference")
print(f"Duration: {TOTAL_DURATION_SEC}s | Batch sizes: {BATCH_SIZES} | Pause: {PAUSE_BETWEEN_SEC}s")
print(f"")
print(f"{'#':>4} {'Batch':>6} {'Latency':>10} {'Throughput':>12} {'Total':>8} {'Elapsed':>8}")
print("-" * 55)

request_log = []
total_predictions = 0
start_time = time.time()
request_num = 0

while (time.time() - start_time) < TOTAL_DURATION_SEC:
    # Pick a random batch size to simulate varying request patterns
    batch_size = BATCH_SIZES[request_num % len(BATCH_SIZES)]
    
    # Sample random rows
    sample_idx = np.random.choice(len(full_df), size=batch_size, replace=False)
    batch_df = full_df.iloc[sample_idx].reset_index(drop=True)
    snow_batch = session.create_dataframe(batch_df)
    
    # Send inference request
    req_start = time.time()
    try:
        result = mv.run(
            snow_batch,
            function_name="predict",
            service_name=SERVICE_NAME,
        )
        result.collect()  # force execution
        req_latency = (time.time() - req_start) * 1000
        status = "OK"
    except Exception as e:
        req_latency = (time.time() - req_start) * 1000
        status = f"ERR: {str(e)[:50]}"
    
    total_predictions += batch_size
    request_num += 1
    elapsed = time.time() - start_time
    throughput = total_predictions / elapsed
    
    request_log.append({
        'request_num': request_num,
        'batch_size': batch_size,
        'latency_ms': round(req_latency, 1),
        'status': status,
        'elapsed_sec': round(elapsed, 1),
        'total_predictions': total_predictions,
    })
    
    # Print every 5th request to avoid flooding output
    if request_num % 5 == 1 or status != "OK":
        print(f"{request_num:>4} {batch_size:>6} {req_latency:>8.0f}ms {throughput:>10.1f}/s {total_predictions:>8} {elapsed:>7.1f}s  {status}")
    
    # Pause to simulate realistic traffic spacing
    time.sleep(PAUSE_BETWEEN_SEC)

total_elapsed = time.time() - start_time
print("-" * 55)
print(f"DONE: {request_num} requests, {total_predictions:,} predictions in {total_elapsed:.0f}s")

In [ ]:
# Analyze traffic simulation results
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

log_df = pd.DataFrame(request_log)
ok_df = log_df[log_df['status'] == 'OK']
err_df = log_df[log_df['status'] != 'OK']

print("=" * 70)
print("TRAFFIC SIMULATION RESULTS")
print("=" * 70)
print(f"Total requests:      {len(log_df)}")
print(f"Successful:          {len(ok_df)} ({len(ok_df)/len(log_df)*100:.0f}%)")
print(f"Failed:              {len(err_df)}")
print(f"Total predictions:   {total_predictions:,}")
print(f"Duration:            {total_elapsed:.0f}s")
print(f"Avg throughput:      {total_predictions/total_elapsed:.1f} predictions/sec")
print(f"")
print(f"Latency (successful requests):")
if len(ok_df) > 0:
    lat = ok_df['latency_ms']
    print(f"  p50:   {lat.quantile(0.50):.0f} ms")
    print(f"  p90:   {lat.quantile(0.90):.0f} ms")
    print(f"  p99:   {lat.quantile(0.99):.0f} ms")
    print(f"  min:   {lat.min():.0f} ms")
    print(f"  max:   {lat.max():.0f} ms")
    print(f"  mean:  {lat.mean():.0f} ms")
print(f"")
print(f"Latency by batch size:")
for bs in sorted(ok_df['batch_size'].unique()):
    subset = ok_df[ok_df['batch_size'] == bs]['latency_ms']
    print(f"  {bs:>3} rows: p50={subset.quantile(0.50):.0f}ms, p99={subset.quantile(0.99):.0f}ms, n={len(subset)}")

# Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Latency over time
axes[0,0].scatter(ok_df['elapsed_sec'], ok_df['latency_ms'], c=ok_df['batch_size'],
                  cmap='Reds', alpha=0.6, s=30, edgecolors='white', linewidths=0.5)
axes[0,0].set_xlabel('Elapsed (sec)')
axes[0,0].set_ylabel('Latency (ms)')
axes[0,0].set_title('Request Latency Over Time', fontweight='bold')
cb = plt.colorbar(axes[0,0].collections[0], ax=axes[0,0])
cb.set_label('Batch size')

# 2. Latency distribution
axes[0,1].hist(ok_df['latency_ms'], bins=30, color='#cc0000', edgecolor='white')
axes[0,1].set_xlabel('Latency (ms)')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Latency Distribution', fontweight='bold')
axes[0,1].axvline(ok_df['latency_ms'].median(), color='black', linestyle='--', label=f"p50={ok_df['latency_ms'].median():.0f}ms")
axes[0,1].legend()

# 3. Cumulative predictions
axes[1,0].plot(ok_df['elapsed_sec'], ok_df['total_predictions'], color='#cc0000', linewidth=2)
axes[1,0].set_xlabel('Elapsed (sec)')
axes[1,0].set_ylabel('Cumulative Predictions')
axes[1,0].set_title('Prediction Throughput', fontweight='bold')
axes[1,0].fill_between(ok_df['elapsed_sec'], ok_df['total_predictions'], alpha=0.1, color='#cc0000')

# 4. Latency by batch size (box plot)
sorted_sizes = sorted(ok_df['batch_size'].unique())
batch_groups = [ok_df[ok_df['batch_size'] == bs]['latency_ms'].values for bs in sorted_sizes]
bp = axes[1,1].boxplot(batch_groups)
axes[1,1].set_xticklabels([str(bs) for bs in sorted_sizes])
axes[1,1].set_xlabel('Batch Size')
axes[1,1].set_ylabel('Latency (ms)')
axes[1,1].set_title('Latency by Batch Size', fontweight='bold')

plt.suptitle('SNOWHEALTH Inference Service - Traffic Simulation Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Service management reference
print("=" * 60)
print("SERVICE MANAGEMENT COMMANDS")
print("=" * 60)
fqn = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{SERVICE_NAME}"

print(f"""
Check status:
  SELECT SYSTEM$GET_SERVICE_STATUS('{fqn}');
  DESCRIBE SERVICE {fqn};

View logs:
  CALL SYSTEM$GET_SERVICE_LOGS('{fqn}', 0, 'model-inference');

Suspend (save cost):
  ALTER SERVICE {fqn} SUSPEND;

Resume:
  ALTER SERVICE {fqn} RESUME;

Scale up:
  ALTER SERVICE {fqn} SET MIN_INSTANCES = 2, MAX_INSTANCES = 4;

Set auto-suspend (seconds):
  ALTER SERVICE {fqn} SET AUTO_SUSPEND_SECS = 600;

Drop service:
  DROP SERVICE {fqn};

SQL inference (from any worksheet):
  SELECT {SERVICE_NAME}!PREDICT(col1, col2, ...) FROM my_table;
""")